# Phase 0: Data Preprocessing & Unification Prototype
This notebook walks through the step-by-step logic used in `src/dental_model/data/`:
1. **Source Data Inspection**: Checking raw archives and folders in `data/raw/`.
2. **Extraction Pipeline**: Unpacking `.zip`, `.rar`, `.7z` datasets to `data/interim/`.
3. **Taxonomy & Label Unification**:
   - Mapping Caries-Spectra categories to unified classes (`caries_early`, `caries_advanced`, `healthy`).
   - Remapping Roboflow YOLO detection labels into standard class indices (0: healthy, 1: plaque, 2: caries).
4. **Stratified Splitting**: 70% train / 15% val / 15% test splits for classifier data.
5. **Data Integrity & Checksums**: Generating SHA-256 hashes in `data/processed/checksums.json`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import yaml

with open('configs/data_paths.yaml') as f:
    cfg = yaml.safe_load(f)

print('Data Sources Config:')
for name, info in cfg['sources'].items():
    print(f' - {name}: status={info.get("status")} | type={info.get("type")}')

## 1. Classifier Label Distribution & Verification

In [ ]:
classifier_csv = Path('data/processed/classifier/labels.csv')
if classifier_csv.exists():
    df = pd.read_csv(classifier_csv)
    print(f'Total classifier samples: {len(df)}')
    print('\nSplit distribution:')
    print(pd.crosstab(df['label'], df['split'], margins=True))
else:
    print('labels.csv not found. Run scripts/prepare_data.ps1')

## 2. Detector Data YAML & Annotation Verification

In [ ]:
detector_yaml = Path('data/processed/detector/data.yaml')
if detector_yaml.exists():
    with open(detector_yaml) as f:
        yolo_cfg = yaml.safe_load(f)
    print('Detector YOLO YAML:')
    print(yaml.dump(yolo_cfg, default_flow_style=False))

    for split in ['train', 'val', 'test']:
        img_count = len(list(Path(f'data/processed/detector/{split}/images').glob('*.*')))
        lbl_count = len(list(Path(f'data/processed/detector/{split}/labels').glob('*.txt')))
        print(f'{split.upper():5s} -> Images: {img_count:5d} | Labels: {lbl_count:5d}')

## 3. Dataset Integrity Checksums

In [ ]:
checksums_file = Path('data/processed/checksums.json')
if checksums_file.exists():
    with open(checksums_file) as f:
        hashes = json.load(f)
    print(f'Total tracked file hashes: {len(hashes):,}')
    print('\nSample entries:')
    for k, v in list(hashes.items())[:5]:
        print(f'  {k} -> {v}')